In [1]:
!pip install pystac-client pystac requests
!pip install laspy[lazrs] pandas numpy
!pip install whitebox
!pip install whitebox --upgrade
!pip install geopandas pandas tqdm pyogrio shapely

# Inventaire des dalles LiDAR HD (IGN)

Ce script interroge l’API STAC de l’IGN pour récupérer automatiquement **toutes les dalles LiDAR HD** couvrant un département donné.  
Il charge le contour du département, interroge l’API, extrait les métadonnées utiles (ID, date de vol, année, mois, URL .laz), filtre les dalles réellement intersectant la zone, puis génère deux fichiers :

- un **GeoPackage** contenant les empreintes géométriques des dalles,
- un **CSV** listant les informations descriptives.

Il permet ainsi de constituer un **inventaire complet et géoréférencé** des données LiDAR HD disponibles pour une zone d’étude.


In [3]:
"""
INVENTAIRE LIDAR HD (IGN)
Objectif : Lister toutes les dalles d'un département via l'API STAC.
Sorties : Un fichier SIG (GeoPackage) et un tableau (CSV).
"""

import pandas as pd
import geopandas as gpd
import requests
import os
from shapely.geometry import shape

# --- CONFIGURATION DES CHEMINS ---
# Dossier où se trouvent vos données et où seront enregistrés les résultats
DOSSIER = r"C:/Users/tliegeon/Desktop/lidar/"

# Fichiers d'entrée et de sortie
FICHIER_CONTOUR = os.path.join(DOSSIER, "gironde.shp")
SORTIE_GPKG = os.path.join(DOSSIER, "Inventaire_LidarHD_Gironde.gpkg")
SORTIE_CSV = os.path.join(DOSSIER, "Inventaire_LidarHD_Gironde.csv")

# Paramètres de l'API IGN
URL_API = "https://api.stac.teledetection.fr/search"
COLLECTION = "lidarhd"

def executer_inventaire():
    """Fonction principale pour générer l'inventaire des dalles."""

    # --- 1. CHARGEMENT DU DÉPARTEMENT ---
    try:
        # Lecture du fichier .shp (en Lambert 93)
        gdf_contour = gpd.read_file(FICHIER_CONTOUR)
        
        # Passage en WGS84 (coordonnées GPS) pour que l'API comprenne la zone
        gdf_wgs84 = gdf_contour.to_crs("EPSG:4326")
        emprise = list(gdf_wgs84.total_bounds)
    except Exception as e:
        print(f"Erreur au chargement du contour : {e}")
        return

    # --- 2. RÉCUPÉRATION DES DALLES (PAR LOTS) ---
    # On utilise une boucle car l'API envoie les dalles par "pages" de 1000 maximum
    print("Recherche des dalles en cours sur le serveur IGN...")
    parametres = {
        "collections": [COLLECTION],
        "bbox": emprise,
        "limit": 1000 
    }
    
    toutes_les_dalles = []
    session = requests.Session()

    while True:
        reponse = session.post(URL_API, json=parametres)
        if reponse.status_code != 200:
            print("Erreur de connexion à l'API.")
            break
            
        donnees = reponse.json()
        toutes_les_dalles.extend(donnees.get('features', []))
        
        # On cherche s'il y a une page suivante pour continuer la collecte
        liens = donnees.get('links', [])
        page_suivante = next((l for l in liens if l.get('rel') == 'next'), None)
        
        if page_suivante and page_suivante.get('method') == 'POST':
            parametres = page_suivante.get('body', parametres)
        else:
            break

    if not toutes_les_dalles:
        print("Aucune dalle trouvée sur cette zone.")
        return

    # --- 3. EXTRACTION DES INFOS (NOM, DATE, LIEN) ---
    print(f"Traitement de {len(toutes_les_dalles)} dalles détectées...")
    liste_finale = []
    
    for dalle in toutes_les_dalles:
        infos = dalle.get('properties', {})
        date_vol = infos.get('start_datetime') or infos.get('datetime')
        
        # Extraction de l'année et du mois
        annee, mois = None, None
        if date_vol:
            try:
                dt = pd.to_datetime(date_vol)
                annee = dt.year
                mois = dt.month
            except:
                pass

        # Récupération du lien de téléchargement du fichier .laz
        fichiers = dalle.get('assets', {})
        url_laz = next((v.get('href') for k, v in fichiers.items() if v.get('href', '').endswith('.laz')), None)
        
        liste_finale.append({
            'nom_complet': dalle.get('id'), # On garde l'ID entier (pas de coupure)
            'annee': annee,
            'mois': mois,
            'date_vol': date_vol,
            'url_laz': url_laz,
            'geometry': shape(dalle['geometry'])
        })

    # --- 4. NETTOYAGE ET FILTRAGE SPATIAL ---
    # Création du tableau géographique
    gdf_resultat = gpd.GeoDataFrame(liste_finale, crs="EPSG:4326")
    
    # On repasse en Lambert 93 pour les besoins cartographiques français
    gdf_resultat = gdf_resultat.to_crs("EPSG:2154")
    
    # On ne garde que les dalles qui touchent vraiment le contour du département
    gdf_final = gpd.sjoin(gdf_resultat, gdf_contour, how="inner", predicate="intersects")

    # Suppression des colonnes inutiles créées par la jointure
    gdf_final = gdf_final.drop(columns=[c for c in ['index_right', 'fid'] if c in gdf_final.columns])

    # --- 5. ENREGISTREMENT DES FICHIERS ---
    try:
        # Enregistrement du fichier spatial
        gdf_final.to_file(SORTIE_GPKG, driver="GPKG")
        
        # Enregistrement du tableau simple
        df_csv = gdf_final.drop(columns='geometry')
        df_csv.to_csv(SORTIE_CSV, index=False, sep=';', encoding='utf-8-sig')
        
        print("\nInventaire terminé")
        print(f"Dalles trouvées : {len(gdf_final)}")
        print(f"Fichiers créés dans : {DOSSIER}")
    except Exception as e:
        print(f"Erreur lors de l'enregistrement : {e}")

if __name__ == "__main__":
    executer_inventaire()

Recherche des dalles en cours sur le serveur IGN...
Traitement de 18042 dalles détectées...

Inventaire terminé
Dalles trouvées : 10759
Fichiers créés dans : C:/Users/tliegeon/Desktop/lidar/
